# Text Network Analysis

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook transforms textual data into **network graphs** that reveal the underlying structure of discourse — identifying influential concepts, thematic clusters, and the pathways through which meaning circulates. Rather than reading through hundreds of pages of field notes, interview transcripts, or documents, you receive a visual map of how concepts relate to one another, which topics dominate the discourse, and where structural gaps suggest potential for new insights.

Building on the text network analysis methodology developed by Paranyushkin (2011, 2019), this notebook represents text as a graph where words are nodes and their co-occurrences are edges. Network analysis algorithms then identify communities of related concepts (themes), measure the influence of individual terms (betweenness centrality), and assess the overall structure of discourse.

## Key Features

- **Multi-Format Input**: Process PDF, DOCX, TXT files, or paste text directly
- **Text Normalization**: Automatic stopword removal, lemmatization, and text cleaning
- **Co-occurrence Network Construction**: Configurable sliding window algorithm for edge creation
- **Community Detection**: Louvain algorithm identifies thematic clusters within discourse
- **Influence Metrics**: Betweenness centrality reveals concepts central to meaning circulation
- **Structural Gap Identification**: Discovers areas where new connections could generate insight
- **Comparative Analysis**: Compare multiple texts to identify shared and divergent themes
- **Multiple Export Formats**: CSV, Excel, JSON, GraphML, and GEXF for Gephi

## Workflow

1. **Configure Parameters**: Set analysis preferences including window size, stopwords, and thresholds
2. **Upload Text(s)**: Single document analysis or batch processing for comparison
3. **Text Preprocessing**: Automatic normalization, lemmatization, and stopword removal
4. **Network Construction**: Build co-occurrence graph using sliding window algorithm
5. **Network Analysis**: Calculate metrics, detect communities, identify influential concepts
6. **Visualization**: Interactive network graphs with community coloring and centrality sizing
7. **Export Results**: Download network data, metrics, and visualizations in multiple formats

## Applications

This notebook supports qualitative research that involves analyzing discourse structure — mapping how concepts relate in interview transcripts, field notes, policy documents, or media coverage. It enables researchers to identify discourse patterns, compare narratives across informants, and discover thematic structures at scale.

## Methodological Positioning

This notebook represents a **computational anthropology** approach — using network analysis to enhance rather than replace traditional qualitative methods. The notebook identifies structural patterns in discourse but leaves interpretation to the researcher. Text network analysis makes visible the pathways for meaning circulation, revealing how concepts cluster and connect.

**Important**: This notebook reveals discourse structure but does not interpret meaning. The researcher's ethnographic knowledge remains essential for understanding why certain concepts cluster together and what structural gaps signify.

## Target Audience

Designed for anthropologists and qualitative researchers working with textual data — from graduate students analyzing interview corpora to research teams examining large document collections for applied projects.

## Technical Approach

The notebook employs **text network analysis** based on word co-occurrence. Text is normalized (stopword removal, lemmatization), then scanned with overlapping windows to create edges between co-occurring terms. The resulting graph is analyzed using network science methods: community detection (Louvain algorithm) identifies thematic clusters, betweenness centrality measures conceptual influence, and modularity assesses discourse structure.

## AI Anthropology Toolkit

This notebook is part of the [AI Anthropology Toolkit](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit), a collection of computational notebooks for AI-assisted qualitative research.

## License & Attribution

This notebook is released under the [PolyForm Noncommercial License 1.0.0](https://polyformproject.org/licenses/noncommercial/1.0.0): free for noncommercial use — research, education, and nonprofit work — with attribution to Matt Artz appreciated. For commercial licensing, contact [Matt Artz](https://www.mattartz.me/).



## Citation

If you use this toolkit in your academic research, please cite:

> Artz, Matt. 2026. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Human Organization. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

Paranyushkin, Dmitry. 2011. "Identifying the Pathways for Meaning Circulation using Text Network Analysis." Nodus Labs.

Paranyushkin, Dmitry. 2019. "InfraNodus: Generating Insight Using Text Network Analysis." In Proceedings of the 2019 World Wide Web Conference (WWW '19). https://doi.org/10.1145/3308558.3314123

## Setup and Installation

*Install required Python packages and import necessary libraries for text processing, network analysis, and visualization. Run this cell first to ensure all dependencies are available.*

In [ ]:
# Install required packages
!pip install networkx python-louvain pandas numpy matplotlib seaborn plotly nltk spacy ipywidgets openpyxl python-docx pypdf2 wordcloud scikit-learn pyvis -q
!python -m spacy download en_core_web_sm -q

import os
import re
import json
import math
import warnings
from datetime import datetime
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional, Set
import io
from pathlib import Path
warnings.filterwarnings('ignore')

# Data processing
import pandas as pd
import numpy as np

# Network analysis
import networkx as nx
import community as community_louvain

# NLP
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
import spacy

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from wordcloud import WordCloud
import plotly.graph_objects as go
from pyvis.network import Network

# File handling
import docx
from PyPDF2 import PdfReader

# UI components
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output, IFrame
from google.colab import files

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Load spaCy model
nlp = spacy.load('en_core_web_sm')

print("✅ All packages installed and imported successfully!")
print(f"📦 NetworkX version: {nx.__version__}")
print(f"📦 spaCy model: en_core_web_sm")

## Configuration

*Define the configuration class and default parameters for text network analysis. These settings control text preprocessing, network construction, and analysis behavior.*

In [ ]:
class TextNetworkConfig:
    """Configuration settings for text network analysis."""

    def __init__(self):
        # Text preprocessing
        self.REMOVE_STOPWORDS = True
        self.LEMMATIZE = True
        self.MIN_WORD_LENGTH = 2
        self.REMOVE_NUMBERS = True
        self.REMOVE_PUNCTUATION = True
        self.LOWERCASE = True
        self.PRESERVE_ENTITIES = False

        # Co-occurrence window settings (based on Paranyushkin methodology)
        self.WINDOW_SIZE_SMALL = 2  # Bigram connections (weight 3)
        self.WINDOW_SIZE_LARGE = 4  # Context connections (weight 1-2)
        self.USE_PARAGRAPH_BOUNDARIES = True
        self.CONNECT_PARAGRAPHS = False

        # Network analysis
        self.MIN_EDGE_WEIGHT = 1
        self.MIN_NODE_FREQUENCY = 1
        self.COMMUNITY_RESOLUTION = 1.0

        # Visualization
        self.MAX_NODES_DISPLAY = 200
        self.NODE_SIZE_METRIC = 'betweenness'
        self.COLOR_BY = 'community'

        # Output
        self.OUTPUT_FOLDER = 'text_network_output'
        self.OUTPUT_FORMAT = 'excel'

        # Custom stopwords
        self.CUSTOM_STOPWORDS = set()

        # Storage
        self.texts = {}
        self.networks = {}
        self.combined_network = None

    def get_stopwords(self):
        """Get combined stopword list."""
        base_stopwords = set(stopwords.words('english'))
        extended = {
            'um', 'uh', 'like', 'know', 'yeah', 'okay', 'ok', 'right',
            'well', 'so', 'just', 'really', 'actually', 'basically',
            'thing', 'things', 'stuff', 'kind', 'sort', 'lot', 'way',
            'going', 'get', 'got', 'go', 'come', 'came', 'say', 'said',
            'think', 'thought', 'want', 'wanted', 'need', 'needed',
            'would', 'could', 'should', 'might', 'may', 'must',
            'one', 'two', 'three', 'first', 'second', 'also', 'even',
            'something', 'anything', 'everything', 'nothing',
            'someone', 'anyone', 'everyone', 'nobody',
            'always', 'never', 'sometimes', 'often', 'usually',
            'yes', 'no', 'maybe', 'perhaps', 'probably'
        }
        return base_stopwords | extended | self.CUSTOM_STOPWORDS

# Initialize global config
config = TextNetworkConfig()
os.makedirs(config.OUTPUT_FOLDER, exist_ok=True)

print("✅ Configuration initialized")
print(f"📁 Output folder: {config.OUTPUT_FOLDER}")

## Text Preprocessing Functions

*Core functions for text normalization including stopword removal, lemmatization, and cleaning. These prepare raw text for network construction while preserving meaningful content.*

In [ ]:
class TextPreprocessor:
    """Handles text normalization and preprocessing for network analysis."""

    def __init__(self, config):
        self.config = config
        self.stopwords = config.get_stopwords()

    def extract_text_from_file(self, file_path: str) -> str:
        """Extract text from various file formats."""
        ext = Path(file_path).suffix.lower()

        if ext == '.txt':
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                return f.read()
        elif ext == '.docx':
            doc = docx.Document(file_path)
            return '\n\n'.join([para.text for para in doc.paragraphs if para.text.strip()])
        elif ext == '.pdf':
            reader = PdfReader(file_path)
            text_parts = [page.extract_text() or '' for page in reader.pages]
            return '\n\n'.join(text_parts)
        else:
            raise ValueError(f"Unsupported file format: {ext}")

    def clean_text(self, text: str) -> str:
        """Basic text cleaning."""
        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'\S+@\S+', '', text)
        text = re.sub(r'[\x00-\x1f\x7f-\x9f]', '', text)
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'\n\s*\n', '\n\n', text)
        return text.strip()

    def tokenize_and_normalize(self, text: str) -> List[List[str]]:
        """Tokenize text into paragraphs of normalized words."""
        if self.config.USE_PARAGRAPH_BOUNDARIES:
            paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
        else:
            paragraphs = [text]

        normalized_paragraphs = []

        for para in paragraphs:
            doc = nlp(para)
            words = []

            for token in doc:
                word = token.lemma_ if self.config.LEMMATIZE else token.text
                if self.config.LOWERCASE:
                    word = word.lower()
                if len(word) < self.config.MIN_WORD_LENGTH:
                    continue
                if self.config.REMOVE_NUMBERS and token.like_num:
                    continue
                if self.config.REMOVE_PUNCTUATION and token.is_punct:
                    continue
                if self.config.REMOVE_STOPWORDS and word in self.stopwords:
                    continue
                if not re.search(r'[a-zA-Z]', word):
                    continue
                words.append(word)

            if words:
                normalized_paragraphs.append(words)

        return normalized_paragraphs

    def process_text(self, text: str) -> Tuple[List[List[str]], Dict]:
        """Complete text preprocessing pipeline."""
        cleaned = self.clean_text(text)
        original_words = len(text.split())
        paragraphs = self.tokenize_and_normalize(cleaned)

        all_words = [w for para in paragraphs for w in para]
        unique_words = set(all_words)
        word_freq = Counter(all_words)

        stats = {
            'original_word_count': original_words,
            'processed_word_count': len(all_words),
            'unique_words': len(unique_words),
            'paragraph_count': len(paragraphs),
            'avg_paragraph_length': np.mean([len(p) for p in paragraphs]) if paragraphs else 0,
            'top_words': word_freq.most_common(20),
            'reduction_ratio': 1 - (len(all_words) / original_words) if original_words > 0 else 0
        }

        return paragraphs, stats

preprocessor = TextPreprocessor(config)
print("✅ Text preprocessor initialized")
print(f"📝 Stopwords loaded: {len(config.get_stopwords())} words")

## Network Construction

*Build the text network graph using the sliding window co-occurrence algorithm. Words become nodes, and edges represent co-occurrence within the specified window sizes, with weights reflecting proximity.*

In [ ]:
class TextNetworkBuilder:
    """Constructs text network graphs from preprocessed text."""

    def __init__(self, config):
        self.config = config

    def build_network(self, paragraphs: List[List[str]]) -> nx.Graph:
        """Build co-occurrence network from normalized paragraphs."""
        G = nx.Graph()
        edge_weights = defaultdict(float)
        node_frequency = Counter()

        for para in paragraphs:
            node_frequency.update(para)

            # First pass: bigram connections (weight = 3)
            for i in range(len(para) - 1):
                w1, w2 = para[i], para[i + 1]
                if w1 != w2:
                    edge_key = tuple(sorted([w1, w2]))
                    edge_weights[edge_key] += 3

            # Second pass: context window connections
            window_size = self.config.WINDOW_SIZE_LARGE
            for i in range(len(para)):
                for j in range(i + 2, min(i + window_size + 1, len(para))):
                    w1, w2 = para[i], para[j]
                    if w1 != w2:
                        distance = j - i
                        weight = max(1, 3 - distance)
                        edge_key = tuple(sorted([w1, w2]))
                        edge_weights[edge_key] += weight

        # Connect paragraphs if enabled
        if self.config.CONNECT_PARAGRAPHS and len(paragraphs) > 1:
            for i in range(len(paragraphs) - 1):
                if paragraphs[i] and paragraphs[i + 1]:
                    last_word = paragraphs[i][-1]
                    first_word = paragraphs[i + 1][0]
                    if last_word != first_word:
                        edge_key = tuple(sorted([last_word, first_word]))
                        edge_weights[edge_key] += 1

        # Build the graph
        for word, freq in node_frequency.items():
            if freq >= self.config.MIN_NODE_FREQUENCY:
                G.add_node(word, frequency=freq)

        for (w1, w2), weight in edge_weights.items():
            if weight >= self.config.MIN_EDGE_WEIGHT:
                if w1 in G.nodes() and w2 in G.nodes():
                    G.add_edge(w1, w2, weight=weight)

        # Remove isolated nodes
        isolated = list(nx.isolates(G))
        G.remove_nodes_from(isolated)

        return G

    def get_network_stats(self, G: nx.Graph) -> Dict:
        """Calculate basic network statistics."""
        if G.number_of_nodes() == 0:
            return {'error': 'Empty network'}

        stats = {
            'nodes': G.number_of_nodes(),
            'edges': G.number_of_edges(),
            'density': nx.density(G),
            'avg_degree': np.mean([d for n, d in G.degree()]),
        }

        if nx.is_connected(G):
            stats['diameter'] = nx.diameter(G)
            stats['avg_path_length'] = nx.average_shortest_path_length(G)
        else:
            largest_cc = max(nx.connected_components(G), key=len)
            subgraph = G.subgraph(largest_cc).copy()
            stats['connected_components'] = nx.number_connected_components(G)
            stats['largest_component_size'] = len(largest_cc)
            stats['largest_component_ratio'] = len(largest_cc) / G.number_of_nodes()
            if len(largest_cc) > 1:
                stats['diameter'] = nx.diameter(subgraph)
                stats['avg_path_length'] = nx.average_shortest_path_length(subgraph)

        stats['avg_clustering'] = nx.average_clustering(G)
        return stats

network_builder = TextNetworkBuilder(config)
print("✅ Network builder initialized")
print(f"🔗 Window sizes: {config.WINDOW_SIZE_SMALL} (bigram), {config.WINDOW_SIZE_LARGE} (context)")

## Network Analysis

*Analyze the text network to identify influential concepts, thematic communities, and discourse structure. Includes centrality measures, community detection, and discourse bias assessment.*

In [ ]:
class TextNetworkAnalyzer:
    """Analyzes text networks to extract insights about discourse structure."""

    def __init__(self, config):
        self.config = config

    def calculate_centrality_measures(self, G: nx.Graph) -> Dict[str, Dict[str, float]]:
        """Calculate various centrality measures for all nodes."""
        centralities = {}
        centralities['betweenness'] = nx.betweenness_centrality(G, weight='weight')
        centralities['degree'] = nx.degree_centrality(G)
        try:
            centralities['eigenvector'] = nx.eigenvector_centrality(G, weight='weight', max_iter=500)
        except:
            centralities['eigenvector'] = {n: 0 for n in G.nodes()}
        centralities['pagerank'] = nx.pagerank(G, weight='weight')
        centralities['strength'] = {n: sum(d['weight'] for _, _, d in G.edges(n, data=True)) for n in G.nodes()}
        return centralities

    def detect_communities(self, G: nx.Graph) -> Tuple[Dict[str, int], float]:
        """Detect communities using Louvain algorithm."""
        partition = community_louvain.best_partition(G, weight='weight', resolution=self.config.COMMUNITY_RESOLUTION)
        modularity = community_louvain.modularity(partition, G, weight='weight')
        return partition, modularity

    def get_community_summary(self, G: nx.Graph, partition: Dict[str, int], centralities: Dict) -> List[Dict]:
        """Summarize each community with top words and statistics."""
        communities = defaultdict(list)
        for node, comm_id in partition.items():
            communities[comm_id].append(node)

        summaries = []
        for comm_id, nodes in sorted(communities.items(), key=lambda x: -len(x[1])):
            bc = centralities['betweenness']
            top_words = sorted(nodes, key=lambda n: bc.get(n, 0), reverse=True)[:5]
            subgraph = G.subgraph(nodes)

            summary = {
                'community_id': comm_id,
                'size': len(nodes),
                'percentage': len(nodes) / G.number_of_nodes() * 100,
                'top_words': top_words,
                'label': ' - '.join(top_words[:3]),
                'internal_edges': subgraph.number_of_edges(),
                'density': nx.density(subgraph) if len(nodes) > 1 else 0,
                'all_words': sorted(nodes, key=lambda n: bc.get(n, 0), reverse=True)
            }
            summaries.append(summary)
        return summaries

    def assess_discourse_structure(self, G: nx.Graph, partition: Dict[str, int], modularity: float, centralities: Dict) -> Dict:
        """Assess discourse structure and bias level."""
        communities = defaultdict(list)
        for node, comm_id in partition.items():
            communities[comm_id].append(node)

        total_nodes = G.number_of_nodes()
        largest_community = max(len(c) for c in communities.values()) if communities else 0
        largest_ratio = largest_community / total_nodes if total_nodes > 0 else 0

        bc = centralities['betweenness']
        top_4_nodes = sorted(bc.keys(), key=lambda x: bc[x], reverse=True)[:4]
        top_communities = [partition.get(n, 0) for n in top_4_nodes]
        comm_counts = Counter(top_communities)
        probs = [c / len(top_4_nodes) for c in comm_counts.values()]
        entropy = -sum(p * math.log2(p) for p in probs if p > 0)

        if modularity > 0.65 and largest_ratio < 0.5 and entropy >= 1.5:
            structure = 'Dispersed'
            description = 'Multiple distinct topics that are weakly related to each other.'
        elif modularity > 0.4 and largest_ratio < 0.5 and entropy >= 1.5:
            structure = 'Diversified'
            description = 'Several balanced topics that are interconnected.'
        elif modularity > 0.2 or (modularity > 0.4 and entropy < 0.75):
            structure = 'Focused'
            description = 'Discourse concentrated around specific perspectives.'
        else:
            structure = 'Biased'
            description = 'Discourse dominated by a single narrative or set of concepts.'

        return {
            'structure_type': structure,
            'description': description,
            'modularity': modularity,
            'largest_community_ratio': largest_ratio,
            'top_concepts_entropy': entropy,
            'num_communities': len(communities),
            'top_4_concepts': top_4_nodes
        }

    def find_structural_gaps(self, G: nx.Graph, partition: Dict[str, int], centralities: Dict, top_n: int = 5) -> List[Dict]:
        """Identify structural gaps - areas where connections are lacking."""
        communities = defaultdict(list)
        for node, comm_id in partition.items():
            communities[comm_id].append(node)

        gaps = []
        comm_ids = list(communities.keys())

        for i, c1 in enumerate(comm_ids):
            for c2 in comm_ids[i + 1:]:
                nodes1 = set(communities[c1])
                nodes2 = set(communities[c2])
                cross_edges = sum(1 for u, v in G.edges() if (u in nodes1 and v in nodes2) or (u in nodes2 and v in nodes1))
                possible_edges = len(nodes1) * len(nodes2)
                connection_ratio = cross_edges / possible_edges if possible_edges > 0 else 0

                bc = centralities['betweenness']
                top1 = sorted(nodes1, key=lambda n: bc.get(n, 0), reverse=True)[:3]
                top2 = sorted(nodes2, key=lambda n: bc.get(n, 0), reverse=True)[:3]

                gaps.append({
                    'community_1': c1,
                    'community_2': c2,
                    'community_1_topics': top1,
                    'community_2_topics': top2,
                    'cross_edges': cross_edges,
                    'connection_ratio': connection_ratio,
                    'gap_score': 1 - connection_ratio
                })

        gaps.sort(key=lambda x: x['gap_score'], reverse=True)
        return gaps[:top_n]

    def analyze_network(self, G: nx.Graph) -> Dict:
        """Complete network analysis pipeline."""
        if G.number_of_nodes() == 0:
            return {'error': 'Empty network'}

        centralities = self.calculate_centrality_measures(G)
        partition, modularity = self.detect_communities(G)
        community_summaries = self.get_community_summary(G, partition, centralities)
        discourse_structure = self.assess_discourse_structure(G, partition, modularity, centralities)
        structural_gaps = self.find_structural_gaps(G, partition, centralities)
        bc = centralities['betweenness']
        top_concepts = sorted(bc.items(), key=lambda x: x[1], reverse=True)[:20]

        return {
            'centralities': centralities,
            'partition': partition,
            'modularity': modularity,
            'communities': community_summaries,
            'discourse_structure': discourse_structure,
            'structural_gaps': structural_gaps,
            'top_concepts': top_concepts
        }

analyzer = TextNetworkAnalyzer(config)
print("✅ Network analyzer initialized")

## Visualization Functions

*Create interactive and static visualizations of the text network including force-directed graphs, community structure diagrams, and metrics charts.*

In [ ]:
class TextNetworkVisualizer:
    """Creates visualizations for text network analysis."""

    def __init__(self, config):
        self.config = config
        self.colors = {
            'primary_bg': '#E7ECEF',
            'primary_text': '#274C77',
            'interactive': '#6096BA',
            'secondary_bg': '#A3CEF1',
            'neutral': '#8B8C89'
        }
        self.community_palette = [
            '#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00',
            '#ffff33', '#a65628', '#f781bf', '#999999', '#66c2a5',
            '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#ffd92f'
        ]

    def get_node_sizes(self, G, centralities, min_size=10, max_size=50):
        """Calculate node sizes based on selected centrality metric."""
        metric = self.config.NODE_SIZE_METRIC
        values = centralities.get(metric, centralities['betweenness'])
        if not values:
            return {n: (min_size + max_size) / 2 for n in G.nodes()}
        min_val, max_val = min(values.values()), max(values.values())
        range_val = max_val - min_val if max_val > min_val else 1
        return {node: min_size + ((values.get(node, min_val) - min_val) / range_val) * (max_size - min_size) for node in G.nodes()}

    def create_static_visualization(self, G, analysis, title="Text Network"):
        """Create static matplotlib visualization."""
        fig, ax = plt.subplots(1, 1, figsize=(14, 10))
        if G.number_of_nodes() == 0:
            ax.text(0.5, 0.5, 'No data to visualize', ha='center', va='center')
            return fig

        # Limit nodes for visualization
        if G.number_of_nodes() > self.config.MAX_NODES_DISPLAY:
            bc = analysis['centralities']['betweenness']
            top_nodes = sorted(bc.keys(), key=lambda x: bc[x], reverse=True)[:self.config.MAX_NODES_DISPLAY]
            G = G.subgraph(top_nodes).copy()

        pos = nx.spring_layout(G, k=2/math.sqrt(G.number_of_nodes()), iterations=50, seed=42)
        partition = analysis['partition']
        node_colors = [self.community_palette[partition.get(n, 0) % len(self.community_palette)] for n in G.nodes()]
        sizes = self.get_node_sizes(G, analysis['centralities'])
        node_sizes = [sizes.get(n, 20) * 10 for n in G.nodes()]
        edge_weights = [G[u][v].get('weight', 1) for u, v in G.edges()]
        max_weight = max(edge_weights) if edge_weights else 1
        edge_widths = [0.5 + (w / max_weight) * 2 for w in edge_weights]

        nx.draw_networkx_edges(G, pos, alpha=0.3, width=edge_widths, edge_color=self.colors['neutral'], ax=ax)
        nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.8, ax=ax)

        bc = analysis['centralities']['betweenness']
        top_nodes = sorted(G.nodes(), key=lambda x: bc.get(x, 0), reverse=True)[:30]
        labels = {n: n for n in top_nodes}
        nx.draw_networkx_labels(G, pos, labels, font_size=8, font_color=self.colors['primary_text'], ax=ax)

        communities = analysis['communities'][:7]
        legend_patches = [mpatches.Patch(color=self.community_palette[c['community_id'] % len(self.community_palette)],
                                         label=f"{c['label']} ({c['size']} words)") for c in communities]
        ax.legend(handles=legend_patches, loc='upper left', fontsize=8, title='Thematic Communities')
        ax.set_title(title, fontsize=14, fontweight='bold', color=self.colors['primary_text'])
        ax.axis('off')
        plt.tight_layout()
        return fig

    def create_interactive_visualization(self, G, analysis, title="Text Network", filename="network.html"):
        """Create interactive PyVis visualization."""
        if G.number_of_nodes() > self.config.MAX_NODES_DISPLAY:
            bc = analysis['centralities']['betweenness']
            top_nodes = sorted(bc.keys(), key=lambda x: bc[x], reverse=True)[:self.config.MAX_NODES_DISPLAY]
            G = G.subgraph(top_nodes).copy()

        net = Network(height='700px', width='100%', bgcolor='#E7ECEF', font_color=self.colors['primary_text'])
        net.force_atlas_2based(gravity=-50, central_gravity=0.01, spring_length=100, spring_strength=0.08)

        partition = analysis['partition']
        sizes = self.get_node_sizes(G, analysis['centralities'], min_size=10, max_size=40)
        bc = analysis['centralities']['betweenness']

        for node in G.nodes():
            comm_id = partition.get(node, 0)
            color = self.community_palette[comm_id % len(self.community_palette)]
            size = sizes.get(node, 15)
            freq = G.nodes[node].get('frequency', 0)
            tooltip = f"<b>{node}</b><br>Community: {comm_id}<br>Frequency: {freq}<br>Betweenness: {bc.get(node, 0):.4f}"
            net.add_node(node, label=node, color=color, size=size, title=tooltip)

        for u, v, data in G.edges(data=True):
            weight = data.get('weight', 1)
            net.add_edge(u, v, value=weight, title=f"Weight: {weight}")

        filepath = os.path.join(self.config.OUTPUT_FOLDER, filename)
        net.save_graph(filepath)
        return filepath

    def create_metrics_dashboard(self, analysis, stats):
        """Create a dashboard of network metrics visualizations."""
        fig = plt.figure(figsize=(16, 12))

        # Centrality distribution
        ax1 = fig.add_subplot(2, 3, 1)
        bc = analysis['centralities']['betweenness']
        ax1.hist(list(bc.values()), bins=30, color=self.colors['interactive'], edgecolor='white', alpha=0.7)
        ax1.set_xlabel('Betweenness Centrality')
        ax1.set_ylabel('Frequency')
        ax1.set_title('Centrality Distribution', fontweight='bold')

        # Community sizes
        ax2 = fig.add_subplot(2, 3, 2)
        communities = analysis['communities']
        labels = [c['label'][:20] + '...' if len(c['label']) > 20 else c['label'] for c in communities[:10]]
        sizes_list = [c['size'] for c in communities[:10]]
        colors_list = [self.community_palette[c['community_id'] % len(self.community_palette)] for c in communities[:10]]
        ax2.barh(labels, sizes_list, color=colors_list)
        ax2.set_xlabel('Number of Words')
        ax2.set_title('Thematic Communities', fontweight='bold')
        ax2.invert_yaxis()

        # Top concepts
        ax3 = fig.add_subplot(2, 3, 3)
        top = analysis['top_concepts'][:15]
        ax3.barh([t[0] for t in top], [t[1] for t in top], color=self.colors['primary_text'])
        ax3.set_xlabel('Betweenness Centrality')
        ax3.set_title('Most Influential Concepts', fontweight='bold')
        ax3.invert_yaxis()

        # Discourse structure info
        ax4 = fig.add_subplot(2, 3, 4)
        ax4.axis('off')
        ds = analysis['discourse_structure']
        info_text = f"Discourse Structure Analysis\n{'='*40}\n\nStructure Type: {ds['structure_type']}\n\n{ds['description']}\n\nMetrics:\n• Modularity: {ds['modularity']:.3f}\n• Communities: {ds['num_communities']}\n• Largest Community: {ds['largest_community_ratio']*100:.1f}%\n• Top Concepts Entropy: {ds['top_concepts_entropy']:.2f}\n\nTop 4 Concepts:\n{', '.join(ds['top_4_concepts'])}"
        ax4.text(0.1, 0.9, info_text, transform=ax4.transAxes, fontsize=10, verticalalignment='top', fontfamily='monospace', bbox=dict(boxstyle='round', facecolor=self.colors['primary_bg'], alpha=0.8))

        # Network statistics
        ax5 = fig.add_subplot(2, 3, 5)
        ax5.axis('off')
        net_stats = f"Network Statistics\n{'='*40}\n\nNodes: {stats.get('nodes', 'N/A')}\nEdges: {stats.get('edges', 'N/A')}\nDensity: {stats.get('density', 0):.4f}\nAverage Degree: {stats.get('avg_degree', 0):.2f}\nAverage Clustering: {stats.get('avg_clustering', 0):.3f}\nDiameter: {stats.get('diameter', 'N/A')}\nAvg Path Length: {stats.get('avg_path_length', 0):.2f}\n\nText Processing:\nOriginal words: {stats.get('original_word_count', 'N/A')}\nProcessed words: {stats.get('processed_word_count', 'N/A')}\nReduction: {stats.get('reduction_ratio', 0)*100:.1f}%"
        ax5.text(0.1, 0.9, net_stats, transform=ax5.transAxes, fontsize=10, verticalalignment='top', fontfamily='monospace', bbox=dict(boxstyle='round', facecolor=self.colors['primary_bg'], alpha=0.8))

        # Structural gaps
        ax6 = fig.add_subplot(2, 3, 6)
        ax6.axis('off')
        gaps = analysis['structural_gaps'][:3]
        gap_text = "Structural Gaps (Insight Potential)\n" + "="*40 + "\n\n"
        for i, gap in enumerate(gaps, 1):
            gap_text += f"{i}. [{', '.join(gap['community_1_topics'][:2])}] <-> [{', '.join(gap['community_2_topics'][:2])}]\n   Gap Score: {gap['gap_score']:.3f}\n\n"
        if not gaps:
            gap_text += "No significant structural gaps detected."
        ax6.text(0.1, 0.9, gap_text, transform=ax6.transAxes, fontsize=10, verticalalignment='top', fontfamily='monospace', bbox=dict(boxstyle='round', facecolor=self.colors['primary_bg'], alpha=0.8))

        plt.suptitle('Text Network Analysis Dashboard', fontsize=16, fontweight='bold', color=self.colors['primary_text'])
        plt.tight_layout()
        return fig

    def create_word_cloud(self, G, centralities):
        """Create word cloud weighted by betweenness centrality."""
        bc = centralities['betweenness']
        word_freq = {word: max(1, int(score * 1000)) for word, score in bc.items()}
        wc = WordCloud(width=1200, height=600, background_color=self.colors['primary_bg'], colormap='viridis', max_words=100, min_font_size=8)
        wc.generate_from_frequencies(word_freq)
        fig, ax = plt.subplots(figsize=(14, 7))
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title('Concept Influence Word Cloud', fontsize=14, fontweight='bold', color=self.colors['primary_text'])
        return fig

visualizer = TextNetworkVisualizer(config)
print("✅ Network visualizer initialized")

## Comparative Analysis

*Functions for comparing multiple text networks to identify shared themes, divergent concepts, and structural differences across documents.*

In [ ]:
class ComparativeAnalyzer:
    """Compare multiple text networks to identify similarities and differences."""

    def __init__(self, config):
        self.config = config

    def compare_networks(self, networks, analyses):
        """Compare multiple text networks."""
        if len(networks) < 2:
            return {'error': 'Need at least 2 networks to compare'}

        node_sets = {name: set(G.nodes()) for name, G in networks.items()}
        all_nodes = set.union(*node_sets.values())
        shared_nodes = set.intersection(*node_sets.values())

        unique_nodes = {}
        for name, nodes in node_sets.items():
            other_nodes = set.union(*[ns for n, ns in node_sets.items() if n != name])
            unique_nodes[name] = nodes - other_nodes

        top_concepts_comparison = {name: [t[0] for t in analysis['top_concepts'][:10]] for name, analysis in analyses.items()}
        structure_comparison = {name: analysis['discourse_structure']['structure_type'] for name, analysis in analyses.items()}
        metrics_comparison = {name: {'nodes': G.number_of_nodes(), 'edges': G.number_of_edges(), 'density': nx.density(G), 'modularity': analyses[name]['modularity'], 'num_communities': len(analyses[name]['communities'])} for name, G in networks.items()}

        return {
            'total_unique_concepts': len(all_nodes),
            'shared_concepts': list(shared_nodes),
            'shared_count': len(shared_nodes),
            'unique_concepts': {k: list(v) for k, v in unique_nodes.items()},
            'top_concepts': top_concepts_comparison,
            'discourse_structures': structure_comparison,
            'metrics': metrics_comparison
        }

    def create_combined_network(self, networks):
        """Create a combined network from multiple texts."""
        combined = nx.Graph()
        for name, G in networks.items():
            for node in G.nodes():
                if node in combined:
                    combined.nodes[node]['sources'].append(name)
                    combined.nodes[node]['frequency'] += G.nodes[node].get('frequency', 1)
                else:
                    combined.add_node(node, sources=[name], frequency=G.nodes[node].get('frequency', 1))
            for u, v, data in G.edges(data=True):
                if combined.has_edge(u, v):
                    combined[u][v]['weight'] += data.get('weight', 1)
                    combined[u][v]['sources'].append(name)
                else:
                    combined.add_edge(u, v, weight=data.get('weight', 1), sources=[name])
        return combined

    def create_comparison_visualization(self, comparison):
        """Visualize comparison results."""
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        metrics = comparison['metrics']
        names = list(metrics.keys())

        # Network size comparison
        ax1 = axes[0, 0]
        x = np.arange(len(names))
        width = 0.35
        ax1.bar(x - width/2, [metrics[n]['nodes'] for n in names], width, label='Nodes', color='#6096BA')
        ax1.bar(x + width/2, [metrics[n]['edges'] for n in names], width, label='Edges', color='#274C77')
        ax1.set_xticks(x)
        ax1.set_xticklabels([n[:15] + '...' if len(n) > 15 else n for n in names], rotation=45, ha='right')
        ax1.legend()
        ax1.set_title('Network Size Comparison', fontweight='bold')

        # Shared vs Unique
        ax2 = axes[0, 1]
        unique_counts = {k: len(v) for k, v in comparison['unique_concepts'].items()}
        labels = ['Shared'] + list(unique_counts.keys())
        values = [comparison['shared_count']] + list(unique_counts.values())
        colors = ['#4daf4a'] + ['#e41a1c', '#377eb8', '#984ea3', '#ff7f00'][:len(unique_counts)]
        ax2.pie(values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
        ax2.set_title('Concept Distribution', fontweight='bold')

        # Modularity comparison
        ax3 = axes[1, 0]
        ax3.bar(names, [metrics[n]['modularity'] for n in names], color='#6096BA')
        ax3.set_xticklabels([n[:15] + '...' if len(n) > 15 else n for n in names], rotation=45, ha='right')
        ax3.set_ylabel('Modularity')
        ax3.set_title('Discourse Modularity', fontweight='bold')
        ax3.axhline(y=0.4, color='red', linestyle='--', alpha=0.5, label='Community threshold')
        ax3.legend()

        # Top concepts text
        ax4 = axes[1, 1]
        ax4.axis('off')
        text = "Top Concepts by Text\n" + "="*40 + "\n\n"
        for name, concepts in comparison['top_concepts'].items():
            short_name = name[:20] + '...' if len(name) > 20 else name
            text += f"{short_name}:\n  {', '.join(concepts[:5])}\n\n"
        text += f"\nShared Top Concepts:\n  {', '.join(comparison['shared_concepts'][:10])}"
        ax4.text(0.1, 0.9, text, transform=ax4.transAxes, fontsize=9, verticalalignment='top', fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='#E7ECEF', alpha=0.8))

        plt.suptitle('Comparative Text Network Analysis', fontsize=14, fontweight='bold')
        plt.tight_layout()
        return fig

comparative_analyzer = ComparativeAnalyzer(config)
print("✅ Comparative analyzer initialized")

## Export Functions

*Export analysis results to multiple formats including CSV, Excel, JSON, and graph formats (GraphML, GEXF) for use with tools like Gephi.*

In [ ]:
class NetworkExporter:
    """Export text network data to various formats."""

    def __init__(self, config):
        self.config = config

    def export_to_gephi(self, G, analysis, filename_base="text_network"):
        """Export network to Gephi-compatible formats."""
        partition = analysis['partition']
        centralities = analysis['centralities']

        for node in G.nodes():
            G.nodes[node]['community'] = partition.get(node, 0)
            G.nodes[node]['betweenness'] = centralities['betweenness'].get(node, 0)
            G.nodes[node]['degree_centrality'] = centralities['degree'].get(node, 0)
            G.nodes[node]['pagerank'] = centralities['pagerank'].get(node, 0)

        gexf_path = os.path.join(self.config.OUTPUT_FOLDER, f"{filename_base}.gexf")
        graphml_path = os.path.join(self.config.OUTPUT_FOLDER, f"{filename_base}.graphml")
        nx.write_gexf(G, gexf_path)
        nx.write_graphml(G, graphml_path)
        return gexf_path, graphml_path

    def export_nodes_table(self, G, analysis):
        """Create a DataFrame of nodes with all metrics."""
        partition = analysis['partition']
        centralities = analysis['centralities']
        comm_labels = {c['community_id']: c['label'] for c in analysis['communities']}

        data = []
        for node in G.nodes():
            comm_id = partition.get(node, 0)
            data.append({
                'word': node,
                'frequency': G.nodes[node].get('frequency', 0),
                'community_id': comm_id,
                'community_label': comm_labels.get(comm_id, f"Community {comm_id}"),
                'degree': G.degree(node),
                'betweenness_centrality': centralities['betweenness'].get(node, 0),
                'degree_centrality': centralities['degree'].get(node, 0),
                'eigenvector_centrality': centralities['eigenvector'].get(node, 0),
                'pagerank': centralities['pagerank'].get(node, 0),
                'strength': centralities['strength'].get(node, 0)
            })
        return pd.DataFrame(data).sort_values('betweenness_centrality', ascending=False)

    def export_edges_table(self, G):
        """Create a DataFrame of edges."""
        return pd.DataFrame([{'source': u, 'target': v, 'weight': d.get('weight', 1)} for u, v, d in G.edges(data=True)]).sort_values('weight', ascending=False)

    def export_communities_table(self, analysis):
        """Create a DataFrame of community information."""
        return pd.DataFrame([{
            'community_id': c['community_id'], 'label': c['label'], 'size': c['size'],
            'percentage': c['percentage'], 'density': c['density'],
            'top_words': ', '.join(c['top_words']), 'all_words': ', '.join(c['all_words'])
        } for c in analysis['communities']])

    def export_structural_gaps_table(self, analysis):
        """Create a DataFrame of structural gaps."""
        return pd.DataFrame([{
            'community_1_id': g['community_1'], 'community_1_topics': ', '.join(g['community_1_topics']),
            'community_2_id': g['community_2'], 'community_2_topics': ', '.join(g['community_2_topics']),
            'cross_edges': g['cross_edges'], 'connection_ratio': g['connection_ratio'], 'gap_score': g['gap_score']
        } for g in analysis['structural_gaps']])

    def export_full_analysis(self, G, analysis, text_stats, network_stats, name="analysis"):
        """Export complete analysis to all formats."""
        paths = {}
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        base_name = f"{name}_{timestamp}"

        nodes_df = self.export_nodes_table(G, analysis)
        edges_df = self.export_edges_table(G)
        communities_df = self.export_communities_table(analysis)
        gaps_df = self.export_structural_gaps_table(analysis)

        # Excel export
        excel_path = os.path.join(self.config.OUTPUT_FOLDER, f"{base_name}.xlsx")
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            summary_data = pd.DataFrame({
                'Metric': ['Total Nodes', 'Total Edges', 'Network Density', 'Average Degree', 'Average Clustering', 'Modularity', 'Number of Communities', 'Discourse Structure', 'Original Word Count', 'Processed Word Count', 'Reduction Ratio'],
                'Value': [network_stats.get('nodes', 0), network_stats.get('edges', 0), f"{network_stats.get('density', 0):.4f}", f"{network_stats.get('avg_degree', 0):.2f}", f"{network_stats.get('avg_clustering', 0):.3f}", f"{analysis['modularity']:.3f}", len(analysis['communities']), analysis['discourse_structure']['structure_type'], text_stats.get('original_word_count', 0), text_stats.get('processed_word_count', 0), f"{text_stats.get('reduction_ratio', 0)*100:.1f}%"]
            })
            summary_data.to_excel(writer, sheet_name='Summary', index=False)
            nodes_df.to_excel(writer, sheet_name='Nodes', index=False)
            edges_df.to_excel(writer, sheet_name='Edges', index=False)
            communities_df.to_excel(writer, sheet_name='Communities', index=False)
            gaps_df.to_excel(writer, sheet_name='Structural_Gaps', index=False)
        paths['excel'] = excel_path

        # CSV export
        csv_nodes_path = os.path.join(self.config.OUTPUT_FOLDER, f"{base_name}_nodes.csv")
        csv_edges_path = os.path.join(self.config.OUTPUT_FOLDER, f"{base_name}_edges.csv")
        nodes_df.to_csv(csv_nodes_path, index=False)
        edges_df.to_csv(csv_edges_path, index=False)
        paths['csv_nodes'] = csv_nodes_path
        paths['csv_edges'] = csv_edges_path

        # Gephi export
        gexf_path, graphml_path = self.export_to_gephi(G.copy(), analysis, base_name)
        paths['gexf'] = gexf_path
        paths['graphml'] = graphml_path

        # JSON export
        json_path = os.path.join(self.config.OUTPUT_FOLDER, f"{base_name}.json")
        json_data = {
            'network_stats': network_stats, 'text_stats': text_stats, 'discourse_structure': analysis['discourse_structure'],
            'top_concepts': analysis['top_concepts'],
            'communities': [{'id': c['community_id'], 'label': c['label'], 'size': c['size'], 'top_words': c['top_words']} for c in analysis['communities']],
            'structural_gaps': analysis['structural_gaps']
        }
        with open(json_path, 'w') as f:
            json.dump(json_data, f, indent=2)
        paths['json'] = json_path

        return paths

exporter = NetworkExporter(config)
print("✅ Network exporter initialized")
print(f"📁 Export formats: Excel, CSV, JSON, GEXF (Gephi), GraphML")

## Configuration Interface

*Interactive widgets for configuring analysis parameters. Adjust settings for text preprocessing, network construction, and visualization.*

In [ ]:
def create_configuration_interface():
    """Create interactive configuration interface."""
    style = {'description_width': '150px'}
    layout = widgets.Layout(width='450px')

    header_html = widgets.HTML("""
    <div style="background: linear-gradient(135deg, #E7ECEF 0%, #A3CEF1 100%);
                padding: 20px; border-radius: 10px; border-left: 5px solid #274C77; margin-bottom: 20px;">
        <h2 style="color: #274C77; margin: 0;">🔗 Text Network Analysis Configuration</h2>
        <p style="color: #274C77; margin: 10px 0 0 0;">Configure parameters for text network construction and analysis.</p>
    </div>
    """)

    preprocess_header = widgets.HTML("<h3 style='margin: 20px 0 10px 0; color: #274C77;'>📝 Text Preprocessing</h3>")

    remove_stopwords = widgets.Checkbox(value=config.REMOVE_STOPWORDS, description='Remove stopwords', style={'description_width': 'initial'}, layout=widgets.Layout(width='300px'))
    lemmatize = widgets.Checkbox(value=config.LEMMATIZE, description='Lemmatize words', style={'description_width': 'initial'}, layout=widgets.Layout(width='300px'))
    min_word_length = widgets.IntSlider(value=config.MIN_WORD_LENGTH, min=1, max=5, step=1, description='Min word length:', style=style, layout=layout)
    custom_stopwords = widgets.Textarea(value='', placeholder='Enter additional stopwords, one per line', description='Custom stopwords:', style=style, layout=widgets.Layout(width='450px', height='80px'))

    network_header = widgets.HTML("<h3 style='margin: 20px 0 10px 0; color: #274C77;'>🔗 Network Construction</h3>")

    window_size = widgets.IntSlider(value=config.WINDOW_SIZE_LARGE, min=2, max=8, step=1, description='Context window:', style=style, layout=layout)
    min_edge_weight = widgets.IntSlider(value=config.MIN_EDGE_WEIGHT, min=1, max=5, step=1, description='Min edge weight:', style=style, layout=layout)
    use_paragraphs = widgets.Checkbox(value=config.USE_PARAGRAPH_BOUNDARIES, description='Respect paragraph boundaries', style={'description_width': 'initial'}, layout=widgets.Layout(width='300px'))

    analysis_header = widgets.HTML("<h3 style='margin: 20px 0 10px 0; color: #274C77;'>📊 Analysis Settings</h3>")

    community_resolution = widgets.FloatSlider(value=config.COMMUNITY_RESOLUTION, min=0.5, max=2.0, step=0.1, description='Community resolution:', style=style, layout=layout)
    node_size_metric = widgets.Dropdown(options=[('Betweenness Centrality', 'betweenness'), ('Degree Centrality', 'degree'), ('PageRank', 'pagerank')], value=config.NODE_SIZE_METRIC, description='Node size by:', style=style, layout=layout)
    max_nodes_display = widgets.IntSlider(value=config.MAX_NODES_DISPLAY, min=50, max=500, step=25, description='Max nodes to show:', style=style, layout=layout)

    status_output = widgets.Output()

    apply_button = widgets.Button(description='✅ Apply Configuration', button_style='primary', layout=widgets.Layout(width='200px', height='40px'))

    def apply_config(b):
        with status_output:
            clear_output()
            config.REMOVE_STOPWORDS = remove_stopwords.value
            config.LEMMATIZE = lemmatize.value
            config.MIN_WORD_LENGTH = min_word_length.value
            config.WINDOW_SIZE_LARGE = window_size.value
            config.MIN_EDGE_WEIGHT = min_edge_weight.value
            config.USE_PARAGRAPH_BOUNDARIES = use_paragraphs.value
            config.COMMUNITY_RESOLUTION = community_resolution.value
            config.NODE_SIZE_METRIC = node_size_metric.value
            config.MAX_NODES_DISPLAY = max_nodes_display.value

            if custom_stopwords.value.strip():
                config.CUSTOM_STOPWORDS = set(w.strip().lower() for w in custom_stopwords.value.split('\n') if w.strip())

            # Reinitialize preprocessor with new config
            global preprocessor
            preprocessor = TextPreprocessor(config)

            print("✅ Configuration applied successfully!")
            print(f"\n📋 Current Settings:")
            print(f"   - Stopword removal: {config.REMOVE_STOPWORDS}")
            print(f"   - Lemmatization: {config.LEMMATIZE}")
            print(f"   - Context window: {config.WINDOW_SIZE_LARGE}")
            print(f"   - Community resolution: {config.COMMUNITY_RESOLUTION}")
            print(f"   - Custom stopwords: {len(config.CUSTOM_STOPWORDS)}")

    apply_button.on_click(apply_config)

    interface = widgets.VBox([
        header_html, preprocess_header,
        widgets.HBox([remove_stopwords, lemmatize]),
        min_word_length, custom_stopwords,
        network_header, window_size, min_edge_weight, use_paragraphs,
        analysis_header, community_resolution, node_size_metric, max_nodes_display,
        widgets.HTML("<br>"), apply_button, status_output
    ])

    display(interface)

create_configuration_interface()

## File Upload

*Upload text files for analysis. Supports PDF, DOCX, and TXT formats, as well as direct text input.*

In [ ]:
def create_upload_interface():
    """Create file upload interface."""
    header_html = widgets.HTML("""
    <div style="background: linear-gradient(135deg, #E7ECEF 0%, #A3CEF1 100%);
                padding: 20px; border-radius: 10px; border-left: 5px solid #274C77; margin-bottom: 20px;">
        <h2 style="color: #274C77; margin: 0;">📁 Upload Text Files</h2>
        <p style="color: #274C77; margin: 10px 0 0 0;">Upload one or more text files for analysis. Supported formats: PDF, DOCX, TXT</p>
    </div>
    """)

    input_mode = widgets.RadioButtons(options=[('Upload files', 'upload'), ('Paste text directly', 'paste')], value='upload', description='Input method:', style={'description_width': '100px'})

    upload_button = widgets.Button(description='📤 Upload Files', button_style='info', layout=widgets.Layout(width='150px', height='40px'))

    text_input = widgets.Textarea(value='', placeholder='Paste your text here...', layout=widgets.Layout(width='100%', height='200px'), disabled=True)
    text_name = widgets.Text(value='pasted_text', description='Text name:', style={'description_width': '100px'}, layout=widgets.Layout(width='300px'), disabled=True)
    add_text_button = widgets.Button(description='➕ Add Text', button_style='success', layout=widgets.Layout(width='120px', height='35px'), disabled=True)

    status_output = widgets.Output()
    files_display = widgets.HTML("<p><i>No files loaded yet.</i></p>")

    def update_input_mode(change):
        if change['new'] == 'paste':
            text_input.disabled = False
            text_name.disabled = False
            add_text_button.disabled = False
            upload_button.disabled = True
        else:
            text_input.disabled = True
            text_name.disabled = True
            add_text_button.disabled = True
            upload_button.disabled = False

    input_mode.observe(update_input_mode, names='value')

    def update_files_display():
        if config.texts:
            html = "<h4 style='color: #274C77;'>📚 Loaded Texts:</h4><ul>"
            for name, text in config.texts.items():
                word_count = len(text.split())
                html += f"<li><b>{name}</b>: {word_count:,} words</li>"
            html += "</ul>"
            files_display.value = html
        else:
            files_display.value = "<p><i>No files loaded yet.</i></p>"

    def handle_upload(b):
        with status_output:
            clear_output()
            print("📤 Please select files to upload...")
            try:
                uploaded = files.upload()
                for filename, content in uploaded.items():
                    temp_path = f"/tmp/{filename}"
                    with open(temp_path, 'wb') as f:
                        f.write(content)
                    text = preprocessor.extract_text_from_file(temp_path)
                    config.texts[filename] = text
                    print(f"✅ Loaded: {filename} ({len(text.split()):,} words)")
                update_files_display()
            except Exception as e:
                print(f"❌ Error: {e}")

    def handle_add_text(b):
        with status_output:
            clear_output()
            if text_input.value.strip():
                name = text_name.value.strip() or f"text_{len(config.texts)+1}"
                config.texts[name] = text_input.value
                print(f"✅ Added: {name} ({len(text_input.value.split()):,} words)")
                text_input.value = ''
                update_files_display()
            else:
                print("⚠️ Please enter some text first.")

    upload_button.on_click(handle_upload)
    add_text_button.on_click(handle_add_text)

    clear_button = widgets.Button(description='🗑️ Clear All', button_style='warning', layout=widgets.Layout(width='120px', height='35px'))

    def handle_clear(b):
        with status_output:
            clear_output()
            config.texts = {}
            config.networks = {}
            update_files_display()
            print("🗑️ All texts cleared.")

    clear_button.on_click(handle_clear)

    interface = widgets.VBox([
        header_html, input_mode,
        widgets.HBox([upload_button, clear_button]),
        widgets.HTML("<hr style='margin: 15px 0;'>"),
        text_name, text_input, add_text_button,
        widgets.HTML("<br>"), files_display, status_output
    ])

    display(interface)

create_upload_interface()

## Run Analysis

*Execute the text network analysis pipeline on uploaded texts. This processes text, builds networks, runs analysis, and generates visualizations.*

In [ ]:
def run_text_network_analysis(single_text_name=None):
    """Run the complete text network analysis pipeline."""
    print("\n" + "="*60)
    print("🔗 TEXT NETWORK ANALYSIS")
    print("="*60)

    if not config.texts:
        print("❌ No texts loaded. Please upload files first.")
        return None

    if single_text_name:
        if single_text_name not in config.texts:
            print(f"❌ Text '{single_text_name}' not found.")
            return None
        texts_to_analyze = {single_text_name: config.texts[single_text_name]}
    else:
        texts_to_analyze = config.texts

    results = {}
    all_analyses = {}
    all_networks = {}

    for name, text in texts_to_analyze.items():
        print(f"\n📝 Processing: {name}")
        print("-" * 40)

        # Step 1: Preprocess text
        print("  ⏳ Preprocessing text...")
        paragraphs, text_stats = preprocessor.process_text(text)
        print(f"  ✅ Processed: {text_stats['processed_word_count']:,} words → {text_stats['unique_words']:,} unique")

        # Step 2: Build network
        print("  ⏳ Building network...")
        G = network_builder.build_network(paragraphs)
        network_stats = network_builder.get_network_stats(G)
        print(f"  ✅ Network: {network_stats['nodes']} nodes, {network_stats['edges']} edges")

        # Step 3: Analyze network
        print("  ⏳ Analyzing network...")
        analysis = analyzer.analyze_network(G)
        print(f"  ✅ Found {len(analysis['communities'])} communities")
        print(f"  ✅ Discourse structure: {analysis['discourse_structure']['structure_type']}")

        # Store results
        config.networks[name] = G
        all_analyses[name] = analysis
        all_networks[name] = G

        results[name] = {
            'network': G,
            'analysis': analysis,
            'text_stats': text_stats,
            'network_stats': network_stats
        }

        # Step 4: Create visualizations
        print("  ⏳ Creating visualizations...")

        # Static network visualization
        fig_network = visualizer.create_static_visualization(G, analysis, f"Text Network: {name}")
        network_path = os.path.join(config.OUTPUT_FOLDER, f"{name}_network.png")
        fig_network.savefig(network_path, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close(fig_network)

        # Metrics dashboard
        combined_stats = {**text_stats, **network_stats}
        fig_dashboard = visualizer.create_metrics_dashboard(analysis, combined_stats)
        dashboard_path = os.path.join(config.OUTPUT_FOLDER, f"{name}_dashboard.png")
        fig_dashboard.savefig(dashboard_path, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close(fig_dashboard)

        # Word cloud
        fig_wordcloud = visualizer.create_word_cloud(G, analysis['centralities'])
        wordcloud_path = os.path.join(config.OUTPUT_FOLDER, f"{name}_wordcloud.png")
        fig_wordcloud.savefig(wordcloud_path, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close(fig_wordcloud)

        # Interactive visualization
        interactive_path = visualizer.create_interactive_visualization(G, analysis, f"Text Network: {name}", f"{name}_interactive.html")

        print(f"  ✅ Visualizations saved")

        # Step 5: Export results
        print("  ⏳ Exporting results...")
        export_paths = exporter.export_full_analysis(G, analysis, text_stats, network_stats, name)
        results[name]['export_paths'] = export_paths
        print(f"  ✅ Exported to: {config.OUTPUT_FOLDER}/")

    # Step 6: Comparative analysis (if multiple texts)
    if len(texts_to_analyze) > 1:
        print("\n" + "="*60)
        print("📊 COMPARATIVE ANALYSIS")
        print("="*60)

        comparison = comparative_analyzer.compare_networks(all_networks, all_analyses)
        results['comparison'] = comparison

        print(f"\n  📈 Total unique concepts: {comparison['total_unique_concepts']}")
        print(f"  🔗 Shared concepts: {comparison['shared_count']}")

        # Create combined network
        combined = comparative_analyzer.create_combined_network(all_networks)
        combined_analysis = analyzer.analyze_network(combined)
        config.combined_network = combined

        # Visualize comparison
        fig_comparison = comparative_analyzer.create_comparison_visualization(comparison)
        comparison_path = os.path.join(config.OUTPUT_FOLDER, "comparison.png")
        fig_comparison.savefig(comparison_path, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close(fig_comparison)

        # Export combined network
        combined_export = exporter.export_full_analysis(
            combined, combined_analysis,
            {'original_word_count': sum(r['text_stats']['original_word_count'] for r in results.values() if 'text_stats' in r),
             'processed_word_count': combined.number_of_nodes(),
             'reduction_ratio': 0},
            network_builder.get_network_stats(combined),
            'combined_network'
        )

        print("  ✅ Comparative analysis complete")

    # Final summary
    print("\n" + "="*60)
    print("✅ ANALYSIS COMPLETE")
    print("="*60)
    print(f"\n📁 All results saved to: {config.OUTPUT_FOLDER}/")
    print("\n📋 Output files include:")
    print("   • Excel workbook with all metrics and data")
    print("   • CSV files (nodes and edges)")
    print("   • GEXF and GraphML files for Gephi")
    print("   • Network visualizations (PNG)")
    print("   • Interactive network (HTML)")
    print("   • JSON data file")

    return results

print("\n✅ Analysis function ready!")
print("\n📌 To run the analysis:")
print("   1. Upload your files using the interface above")
print("   2. Configure settings if needed")
print("   3. Run: results = run_text_network_analysis()")

## Execute Analysis

*Run the analysis on your uploaded texts.*

In [ ]:
# Run the complete text network analysis
results = run_text_network_analysis()

## View Results

*Display and explore the analysis results including visualizations and key findings.*

In [ ]:
def display_results(results):
    """Display analysis results with visualizations."""
    if results is None:
        print("No results to display. Please run the analysis first.")
        return

    for name, data in results.items():
        if name == 'comparison':
            continue

        print(f"\n{'='*60}")
        print(f"📊 Results for: {name}")
        print(f"{'='*60}")

        analysis = data['analysis']

        # Top concepts
        print("\n🎯 Most Influential Concepts:")
        for word, score in analysis['top_concepts'][:10]:
            print(f"   • {word}: {score:.4f}")

        # Discourse structure
        ds = analysis['discourse_structure']
        print(f"\n📈 Discourse Structure: {ds['structure_type']}")
        print(f"   {ds['description']}")

        # Communities
        print("\n🏘️ Thematic Communities:")
        for comm in analysis['communities'][:5]:
            print(f"   • {comm['label']} ({comm['size']} words, {comm['percentage']:.1f}%)")

        # Structural gaps
        if analysis['structural_gaps']:
            print("\n🔍 Structural Gaps (Insight Opportunities):")
            for gap in analysis['structural_gaps'][:3]:
                c1 = ', '.join(gap['community_1_topics'][:2])
                c2 = ', '.join(gap['community_2_topics'][:2])
                print(f"   • [{c1}] ↔ [{c2}] (gap score: {gap['gap_score']:.3f})")

    # Comparison results
    if 'comparison' in results:
        print(f"\n{'='*60}")
        print("📊 COMPARATIVE ANALYSIS")
        print(f"{'='*60}")

        comp = results['comparison']
        print(f"\n🔗 Shared concepts across all texts: {comp['shared_count']}")
        if comp['shared_concepts']:
            print(f"   Top shared: {', '.join(comp['shared_concepts'][:10])}")

        print("\n📊 Discourse structures:")
        for name, structure in comp['discourse_structures'].items():
            print(f"   • {name}: {structure}")

# Display results if available
if 'results' in dir() and results is not None:
    display_results(results)
else:
    print("Run the analysis first to see results.")

## Download Results

*Download all analysis outputs as a ZIP file.*

In [ ]:
import shutil

def download_results():
    """Create a ZIP file of all results and trigger download."""
    if not os.path.exists(config.OUTPUT_FOLDER) or not os.listdir(config.OUTPUT_FOLDER):
        print("❌ No results to download. Please run the analysis first.")
        return

    # Create ZIP file
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    zip_name = f"text_network_results_{timestamp}"
    zip_path = shutil.make_archive(zip_name, 'zip', config.OUTPUT_FOLDER)

    print(f"📦 Created: {zip_path}")
    print("📥 Starting download...")

    # Trigger download
    files.download(zip_path)

    print("\n✅ Download initiated!")
    print("\n📋 ZIP contents:")
    for f in os.listdir(config.OUTPUT_FOLDER):
        print(f"   • {f}")

# Run download
download_results()

## Interactive Network Viewer

*View the interactive network visualization in the notebook.*

In [ ]:
def show_interactive_network(text_name=None):
    """Display interactive network visualization."""
    if text_name is None and config.texts:
        text_name = list(config.texts.keys())[0]

    if text_name is None:
        print("❌ No text loaded. Please upload and analyze text first.")
        return

    html_path = os.path.join(config.OUTPUT_FOLDER, f"{text_name}_interactive.html")

    if os.path.exists(html_path):
        print(f"🔗 Interactive Network: {text_name}")
        print("\n💡 Tip: Hover over nodes to see details. Drag to rearrange.")
        display(IFrame(src=html_path, width=1000, height=700))
    else:
        print(f"❌ Interactive visualization not found for '{text_name}'.")
        print("   Please run the analysis first.")

# Show interactive network for first loaded text
show_interactive_network()